In [ ]:
from datetime import datetime
import pandas as pd
from matplotlib import pyplot as plt
import pycountry
from os import listdir as ls
import numpy as np

from emu_renewal.inputs import OUTPUTS_PATH, DATA_PATH, get_google_mobility
from emu_renewal.plotting import compare_proc_versus_mobility, MOB_COLOURS
from emu_renewal.utils import get_countries_by_continent

In [ ]:
job_path = OUTPUTS_PATH / "44255911"
all_countries = ls(job_path)
countries_by_cont = get_countries_by_continent(all_countries)

In [ ]:
def compare_proc_mob(countries, n_cols, mob_type):
    n_rows = int(np.ceil(len(countries) / n_cols))
    height = 2.0 + n_rows * 2.5
    fig, axes = plt.subplots(n_rows, n_cols, figsize=[12, height])
    title = f"Comparison of variable process to {mob_type.replace('_', ' ')} mobility"
    fig.suptitle(title, fontsize=20, y=1.0)
    flat_axes = axes.ravel()
    for c, country in enumerate(countries):
        ax = flat_axes[c]
        country_name = pycountry.countries.lookup(country).name
        plt.setp(ax.xaxis.get_majorticklabels(), rotation=70)
        
        proc_samples = pd.read_hdf(job_path / country / "no_mob/spaghetti.h5")["process"]
        centiles = proc_samples.quantile([0.05, 0.5, 0.95], axis=1).T
        ax.plot(centiles.index, centiles[0.5], label="process", color="navy")
        ax.fill_between(centiles.index, centiles[0.05], centiles[0.95], alpha=0.2, color="navy")

        try:
            mob = get_google_mobility(country)
            mobility = mob.loc[(centiles.index[0] < mob.index) & (mob.index < centiles.index[-1])]
            smoothed_mob = mobility.rolling(7, center=True).mean().dropna()
            ax.plot(smoothed_mob.index, smoothed_mob[mob_type], color=MOB_COLOURS["g_mob"])
            ax.set_title(country_name)
        except:
            ax.set_title(f"{country_name} (mobility data unavailable)")
    for a in range(c + 1, len(flat_axes)):
        ax = flat_axes[a]
        ax.set_axis_off()
        
    fig.tight_layout()

In [ ]:
mob_domain_map = {
    "retail_and_recreation": "g_mob",
    "grocery_and_pharmacy": "g_mob",
    "parks": "g_mob",
    "transit_stations": "g_mob",
    "workplaces": "g_mob",
    "residential": "g_mob",
    "driving": "a_mob",
    "transit": "a_mob",
    "walking": "a_mob",
    "fb_mob": "fb_mob",
}    

In [ ]:
for mob_type in list(get_google_mobility("AUS").columns):
    compare_proc_mob(countries_by_cont["NA"], 3, mob_type)
    compare_proc_mob(countries_by_cont["SA"], 3, mob_type)
    compare_proc_mob(countries_by_cont["EU"][:16], 4, mob_type)
    compare_proc_mob(countries_by_cont["EU"][16:], 4, mob_type)
    compare_proc_mob(countries_by_cont["AF"], 4, mob_type)
    compare_proc_mob(countries_by_cont["AS"], 4, mob_type)
    compare_proc_mob(countries_by_cont["OC"][:16], 4, mob_type)